# PlantMD — Training on Google Colab
### By Paul Sentongo

This notebook lets you train the full PlantMD model on Google Colab's **free NVIDIA GPU** when you don't have CUDA on your local machine.

The workflow is:
1. Mount Google Drive (so nothing is lost when Colab disconnects)
2. Clone the repository from GitHub into Drive
3. Install dependencies
4. Set up Kaggle credentials and download the dataset
5. Train the model (GPU-accelerated)
6. Upload the trained weights to Hugging Face Hub
7. Optionally deploy the Space

**Before you start:**
- Go to Runtime → Change runtime type → Select **T4 GPU**
- Have your Kaggle API key ready (kaggle.json)
- Have your Hugging Face write token ready (from huggingface.co/settings/tokens)

In [ ]:
# ── Step 0: Verify we have a GPU ──────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅  GPU detected: {gpu}  ({vram:.1f} GB VRAM)')
else:
    print('❌  No GPU found.')
    print('Go to: Runtime → Change runtime type → Hardware accelerator → T4 GPU')
    raise RuntimeError('Please enable GPU and re-run.')

In [ ]:
# ── Step 1: Mount Google Drive ────────────────────────────────────────────────
# Why Google Drive?
# Colab VMs are ephemeral — everything in /content/ is wiped when the session
# ends or times out.  By mounting Drive we persist the dataset, checkpoints,
# and the final model so we never have to re-download or re-train from scratch.

from google.colab import drive
drive.mount('/content/drive')

# We'll work inside a dedicated folder in Drive.
# Change this path if you prefer a different location.
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/PlantMD'

import os
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
print(f'Working directory: {DRIVE_PROJECT_DIR}')

In [ ]:
# ── Step 2: Clone or update the repository ────────────────────────────────────
# If the repo is already in Drive, we just pull the latest changes.
# Otherwise we clone it fresh.
#
# Replace the URL below with your own GitHub repository URL.

GITHUB_REPO_URL = 'https://github.com/Sentoz/Plant-Disease-Classifier.git'
REPO_DIR = f'{DRIVE_PROJECT_DIR}/Plant-Disease-Classifier'

if os.path.exists(f'{REPO_DIR}/.git'):
    print('Repository already cloned.  Pulling latest changes ...')
    !cd {REPO_DIR} && git pull
else:
    print('Cloning repository ...')
    !git clone {GITHUB_REPO_URL} {REPO_DIR}

# Change into the project directory for all subsequent commands
%cd {REPO_DIR}
print(f'Current directory: {os.getcwd()}')

In [ ]:
# ── Step 3: Install dependencies ──────────────────────────────────────────────
# On Colab we have a CUDA GPU, so we install the standard CUDA torch wheels.
# Colab already has PyTorch pre-installed, but we ensure the exact versions
# we need are present.
#
# We skip torch/torchvision/torchaudio from requirements.txt (Colab has them)
# and install only the other packages.  This saves 5-10 minutes.

!pip install timm torchinfo mlflow streamlit kagglehub huggingface_hub \
             seaborn plotly PyYAML python-dotenv rich tqdm \
             scikit-learn --quiet

# Install the project itself as an editable package
!pip install -e . --quiet

import importlib, sys
# Reload sys.path to pick up the newly installed packages
import site
importlib.invalidate_caches()

print('\n✅  Dependencies installed.')

In [ ]:
# ── Step 4: Kaggle credentials (NO JSON FILE NEEDED) ─────────────────────────
#
# IMPORTANT: After this test run, go to kaggle.com/settings → API → Expire Token
# and generate a fresh one.  Store the new one in Colab Secrets instead.
#
# HOW TO ADD YOUR KAGGLE KEY TO COLAB SECRETS (permanent fix, do once):
# ─────────────────────────────────────────────────────────────────────
# 1. Left sidebar in Colab → click the 🔑 KEY icon
# 2. Add secret:  KAGGLE_USERNAME  =  sentongogray
# 3. Add secret:  KAGGLE_KEY       =  (your new token after revoking this one)
# ─────────────────────────────────────────────────────────────────────────────

import os, json

try:
    # PRIMARY: Colab Secrets (once you've added them to the 🔑 panel)
    from google.colab import userdata
    kaggle_username = userdata.get('KAGGLE_USERNAME')
    kaggle_key      = userdata.get('KAGGLE_KEY')

    if not kaggle_username or not kaggle_key:
        raise ValueError("Secrets not set yet")

    print(f'✅  Loaded from Colab Secrets  (user: {kaggle_username})')

except Exception:
    # FALLBACK: hardcoded for this test run
    # ⚠️  Revoke this token after testing: kaggle.com/settings → API → Expire Token
    kaggle_username = 'sentongogray'
    kaggle_key      = 'KGAT_efbb6e7a0b45e4396a06f644efb44c22'
    print(f'✅  Using hardcoded credentials  (user: {kaggle_username})')
    print('⚠️  Remember to revoke this token after testing and move to Colab Secrets.')

# Write to ~/.kaggle/kaggle.json — both kagglehub and kaggle-cli read from here
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
kaggle_path = os.path.expanduser('~/.kaggle/kaggle.json')

with open(kaggle_path, 'w') as f:
    json.dump({'username': kaggle_username, 'key': kaggle_key}, f)

os.chmod(kaggle_path, 0o600)

# Also set as environment variables (kagglehub checks these too)
os.environ['KAGGLE_USERNAME'] = kaggle_username
os.environ['KAGGLE_KEY']      = kaggle_key

print(f'Kaggle credentials written to {kaggle_path}')

In [ ]:
# ── Step 5: Download the dataset ──────────────────────────────────────────────
# kagglehub caches downloads in ~/.cache/kagglehub/.
# We symlink the cache into Drive so the 3 GB download is reused across sessions.

import subprocess

KAGGLE_CACHE = f'{DRIVE_PROJECT_DIR}/kaggle_cache'
os.makedirs(KAGGLE_CACHE, exist_ok=True)

# Point kagglehub cache to Drive so re-runs are instant
os.environ['KAGGLE_CACHE_FOLDER'] = KAGGLE_CACHE
# Alternative for some versions:
os.environ['KAGGLEHUB_CACHE'] = KAGGLE_CACHE

print('Downloading dataset (first time ~3 GB, subsequent runs use cache) ...')
import sys
sys.path.insert(0, REPO_DIR)

from src.data.download import download_dataset
from src.utils.config import load_config

config = load_config()
dataset_path = download_dataset(config)
print(f'\n✅  Dataset ready at: {dataset_path}')

In [ ]:
# ── Step 6: Override config for Colab (optional) ──────────────────────────────
# On Colab T4 (16 GB VRAM) we can use a larger batch size than the default.
# Edit these values to tune for speed vs accuracy.

import yaml

config_path = f'{REPO_DIR}/configs/config.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

# Colab-optimised settings
config['training']['batch_size']     = 64   # T4 has 16 GB, can handle bigger batches
config['training']['num_workers']    = 4    # Colab has 4 vCPUs
config['training']['use_amp']        = True # Always use AMP on GPU
config['training']['epochs']         = 30   # Adjust as needed

# Save the updated config
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print('Config updated for Colab:')
print(f"  batch_size  : {config['training']['batch_size']}")
print(f"  num_workers : {config['training']['num_workers']}")
print(f"  use_amp     : {config['training']['use_amp']}")
print(f"  epochs      : {config['training']['epochs']}")

In [ ]:
# ── Step 7: Train the model ───────────────────────────────────────────────────
# We call the training script directly from Python rather than using !python
# because this keeps all output visible in the notebook cell and also means
# a Colab timeout during training (which sometimes happens) does not kill
# the Python process.
#
# Estimated time on Colab T4:
#   30 epochs, batch_size=64 → roughly 2–3 hours
#   5 epochs  (quick test)   → roughly 20–30 minutes

import runpy

# Pass arguments by modifying sys.argv
import sys
sys.argv = [
    'train.py',
    '--skip_download',    # dataset is already downloaded
    # '--epochs', '5',    # uncomment for a quick test run
]

# Run train.py as __main__
runpy.run_path(f'{REPO_DIR}/train.py', run_name='__main__')

In [ ]:
# ── Step 8: Save the trained model to Drive ───────────────────────────────────
# Copy the best model to a persistent location in Drive so it survives
# Colab session resets.

import shutil

src_model  = f'{REPO_DIR}/models/best_model.pth'
drive_model = f'{DRIVE_PROJECT_DIR}/best_model.pth'

if os.path.exists(src_model):
    shutil.copy2(src_model, drive_model)
    size_mb = os.path.getsize(drive_model) / 1e6
    print(f'✅  Model saved to Drive: {drive_model}  ({size_mb:.1f} MB)')
else:
    print('❌  Model file not found.  Did training complete successfully?')

In [ ]:
# ── Step 9: Upload to Hugging Face Hub ───────────────────────────────────────
# Now we push the trained weights to HF Hub so the Streamlit app on
# HF Spaces can download them at startup.
#
# You need a write-access token from: https://huggingface.co/settings/tokens

from getpass import getpass

HF_TOKEN = getpass('Enter your Hugging Face write token (hf_...): ')
os.environ['HF_TOKEN'] = HF_TOKEN

from src.utils.model_hub import upload_model_to_hub

url = upload_model_to_hub(
    local_model_path=src_model,
    token=HF_TOKEN,
)
print(f'\n✅  Model uploaded to Hugging Face Hub')
print(f'    URL: {url}')

In [ ]:
# ── Step 10 (optional): Deploy the Streamlit Space ───────────────────────────
# Push the app code to HF Spaces.  This step is optional from Colab;
# you can also push from your local machine with: python deploy.py --model-only
# (you already uploaded the model above, so only the Space push is new)

DEPLOY = True   # set to False to skip

if DEPLOY:
    import sys
    sys.argv = ['deploy.py', '--model-only']  # model already uploaded above
    # To also push the Space code, use: sys.argv = ['deploy.py']

    runpy.run_path(f'{REPO_DIR}/deploy.py', run_name='__main__')

## After Training

Once training completes:

1. The model weights are in Google Drive at `PlantMD/best_model.pth`
2. The weights have been uploaded to your HF Hub model repo
3. Your HF Space (if deployed) will automatically download the weights on first access

**To resume a disconnected session:**
- Run Steps 1-3 again (clone/install)
- Copy `best_model.pth` from Drive back to `models/` in the repo
- You can then continue fine-tuning from the saved checkpoint

**To download the model back to your local machine:**
```python
from src.utils.model_hub import download_model_from_hub
path = download_model_from_hub(local_dir='models/')
print('Model at:', path)
```

**Common Colab issues:**

- *Session disconnects during training* — Colab free tier disconnects after ~90 min of inactivity. The model checkpoint saved to Drive is safe. Re-run from Step 1; training can resume from the last checkpoint.
- *CUDA out of memory* — Reduce `batch_size` in Step 6 (try 32 instead of 64).
- *Kaggle download fails* — Re-run the credentials cell (Step 4). The `.kaggle/kaggle.json` file does not persist across Colab sessions unless you save it to Drive.
- *import errors* — Run `%cd {REPO_DIR}` and `import sys; sys.path.insert(0, '.')` to fix path issues.